# 3.3 · Four candidate signals for the long-only portfolio

Notebook 3.2 ended with a portfolio that tracks the market with smaller drawdowns and a higher Sharpe ratio, but adds no
return over SPTM: its five factors (value, quality, momentum, low volatility, net issuance) select stocks no better
than holding the whole universe. This notebook tests four candidate signals, each added to that portfolio on its own:
first two that need no new data and that the literature finds
to capture different effects from those five:

| Signal | Economic intuition | Measurement |
|---|---|---|
| **Earnings surprise** (SUE) | prices under-react to earnings news and drift in its direction for weeks afterwards (*post-earnings-announcement drift*, Ball and Brown 1968; Bernard and Thomas 1989) | standardised unexpected earnings: the latest quarter's earnings minus the same quarter a year earlier, less the company's average such change, divided by the standard deviation of those changes |
| **Residual momentum** | momentum in the stock-specific part of returns is more persistent and less crash-prone than in total returns (Blitz, Huij and Martens, 2011) | the past 12-month return (excluding the latest month) net of what the market factor explains, divided by the volatility of the residuals |

Conventional momentum partly ranks on *beta*: after a rising market, high-beta stocks lead the ranking, and they fall
hardest when the market turns (the *momentum crashes* of 2009 and 2020). Residual momentum removes the market
component first, so it should suffer less in those reversals.

**Test design.** Each signal changes the second version of notebook 3.2 in exactly one way - earnings surprise is added,
residual momentum replaces conventional momentum - and every run is registered. The criterion is the **active
return** against SPTM and its deflated Sharpe ratio, not the strategy's own Sharpe ratio (notebook 3.2 explains why).

In [1]:
from dataclasses import replace

import pandas as pd
from IPython.display import display

from equity_research import charts
from equity_research.db.engine import get_engine
from equity_research.research.backtest import decile_returns
from equity_research.research.cache import ResearchCache
from equity_research.research.data import load_universe_data
from equity_research.research.indices import calendar_years, vendor_returns
from equity_research.research.strategy import StrategyConfig, factor_scores, run_strategy

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")

charts.style()
cache = ResearchCache()
engine = get_engine()
FIVE = (("value", 1.0), ("quality", 1.0), ("momentum", 1.0), ("low_vol", 1.0), ("issuance", 1.0))
second = StrategyConfig(
    family="multi_factor_lo",
    universe="SP1500",
    start="2010-01-01",
    end="2024-12-31",
    factors=FIVE,
    enter=0.30,
    exit_=0.40,
    long_only=True,
    beta_neutral=False,
    weighting="market_cap",
)
data = load_universe_data(cache, second.universe, second.start, second.end)
sptm = vendor_returns(cache, "SPTM", total=True)

## Decile analysis of each signal

Every month the constituents are sorted into deciles by one score, and each decile's equal-weighted return over the
following month is compared with the universe average (before costs). A long-only portfolio can only exploit the
**top** decile; a score that mainly identifies losers shows up in the **bottom** one. The *t*-statistic tests whether
the top decile's average excess return differs from zero: about 2 or more is significant at the 5 % level, below 1 is
indistinguishable from noise.

In [2]:
SIGNALS = ["value", "quality", "momentum", "low_vol", "issuance", "earnings_surprise", "residual_momentum"]
scores = {name: factor_scores(data, name) for name in SIGNALS}
deciles = {name: decile_returns(scores[name], data.returns, data.members) for name in SIGNALS}
alone = pd.DataFrame(
    {
        name: {
            "bottom tenth, a year": d.loc[1, "excess"],
            "top tenth, a year": d.loc[10, "excess"],
            "top tenth t": d.loc[10, "t"],
        }
        for name, d in deciles.items()
    }
).T
display(
    alone.style.format({"bottom tenth, a year": "{:+.1%}", "top tenth, a year": "{:+.1%}", "top tenth t": "{:.1f}"})
)

alike = pd.DataFrame(
    {new: {old: scores[new].corrwith(scores[old], axis=1).mean() for old in SIGNALS[:5]} for new in SIGNALS[5:]}
)
print("How alike the new scores are to the old ones (average correlation across companies each month):")
display(alike.style.format("{:+.2f}"))

,"bottom tenth, a year","top tenth, a year",top tenth t
value,+0.3%,+2.2%,0.9
quality,-2.6%,+0.6%,0.4
momentum,-3.0%,+1.2%,0.5
low_vol,-0.1%,-1.7%,-0.6
issuance,-3.9%,+2.8%,1.9
earnings_surprise,-0.2%,+0.0%,0.0
residual_momentum,-0.9%,+1.9%,1.0


How alike the new scores are to the old ones (average correlation across companies each month):


,earnings_surprise,residual_momentum
value,-0.07,-0.10
quality,+0.01,-0.02
momentum,+0.22,+0.71
low_vol,+0.02,+0.08
issuance,-0.05,-0.00


## Inside the strategy

The second version of notebook 3.2, then the same with one change each. A final run tests the residual-momentum version
from 2006, through the 2008 crisis - years that played no part in its design, so an approximately out-of-sample test.

In [3]:
residual = tuple(("residual_momentum", w) if name == "momentum" else (name, w) for name, w in FIVE)
configs = {
    "Second version (notebook 3.2)": second,
    "+ earnings surprise": replace(second, factors=(*FIVE, ("earnings_surprise", 1.0))),
    "residual momentum instead of momentum": replace(second, factors=residual),
    "residual momentum, from 2006": replace(second, factors=residual, start="2006-01-01"),
}
runs = {name: run_strategy(cache, cfg, engine=engine) for name, cfg in configs.items()}
ACTIVE = {
    "cagr": "{:.1%}",
    "sharpe": "{:.2f}",
    "max_drawdown": "{:.0%}",
    "beta": "{:.2f}",
    "active_return": "{:+.1%}",
    "tracking_error": "{:.1%}",
    "information_ratio": "{:+.2f}",
    "deflated_active_sharpe": "{:.0%}",
}
display(pd.DataFrame({name: r.metrics for name, r in runs.items()}).T[list(ACTIVE)].style.format(ACTIVE))
print(
    "Registered as research runs",
    [r.run_id for r in runs.values()],
    "- trials of this family so far:",
    max(r.metrics["trials"] for r in runs.values()),
)

,cagr,sharpe,max_drawdown,beta,active_return,tracking_error,information_ratio,deflated_active_sharpe
Second version (notebook 3.2),13.6%,0.87,-34%,0.89,-0.3%,3.7%,-0.14,1%
+ earnings surprise,14.1%,0.89,-34%,0.90,+0.1%,3.8%,-0.02,4%
residual momentum instead of momentum,12.9%,0.84,-34%,0.87,-1.0%,4.1%,-0.29,0%
"residual momentum, from 2006",9.9%,0.63,-45%,0.84,-0.4%,4.4%,-0.17,1%


Registered as research runs [45, 46, 47, 48] - trials of this family so far: 12


In [4]:
ahead = pd.DataFrame(
    {
        name: calendar_years({name: r.backtest.net, "SPTM": sptm}).pipe(lambda y, n=name: y[n] - y["SPTM"])
        for name, r in runs.items()
    }
)
print("Each year: the strategy's return minus SPTM's")
display(ahead.style.format("{:+.1%}", na_rep=""))

Each year: the strategy's return minus SPTM's


,Second version (notebook 3.2),+ earnings surprise,residual momentum instead of momentum,"residual momentum, from 2006"
trade_date,,,,
2006,,,,+3.0%
2007,,,,-2.9%
2008,,,,+9.6%
2009,,,,-7.7%
2010,-6.3%,-4.2%,-5.7%,-6.7%
2011,+7.3%,+6.6%,+6.9%,+6.7%
2012,-0.5%,-0.9%,-2.7%,-2.8%
2013,-1.3%,-1.0%,-1.1%,-1.1%
2014,-0.1%,+0.2%,-0.5%,-0.5%


## Two cash-flow signals

Accrual accounting gives management discretion over reported earnings; cash flows are harder to manage. Two scores
built from the cash-flow statement, which the SEC's filings carry and which were added to the accounts for this test:

| Signal | Economic intuition | Measurement |
|---|---|---|
| **Free-cash-flow yield** | cash generation relative to price is a cleaner value measure than earnings | (operating cash flow - capital expenditure) / enterprise value |
| **Accruals** (low is good) | earnings far above cash flow tend not to persist, and the market overweights them (Sloan, 1996) | (net income - operating cash flow) / total assets |

When this notebook was run, LSEG's statements had no cash-flow items, so a score existed only on dates whose accounts
came from the SEC - from about 2010-2011, once a company's XBRL filings covered four quarters. The table shows the
fraction of constituents with a score. (LSEG's annual cash flows were loaded on 2026-10-31; a rerun would cover the
earlier years too.)

In [5]:
CASH = ["cash_flow_yield", "accruals"]
for name in CASH:
    scores[name] = factor_scores(data, name)
members_on = data.members.reindex(data.rebalances)
covered = pd.DataFrame(
    {name: (scores[name].notna() & members_on).sum(axis=1) / members_on.sum(axis=1) for name in CASH}
)
display(covered[covered.index.month == 12].style.format("{:.0%}"))
cash_alone = pd.DataFrame(
    {
        name: {
            "bottom tenth, a year": d.loc[1, "excess"],
            "top tenth, a year": d.loc[10, "excess"],
            "top tenth t": d.loc[10, "t"],
        }
        for name in CASH
        for d in [decile_returns(scores[name], data.returns, data.members)]
    }
).T
display(
    cash_alone.style.format(
        {"bottom tenth, a year": "{:+.1%}", "top tenth, a year": "{:+.1%}", "top tenth t": "{:.1f}"}
    )
)
print("How alike they are to the old scores:")
display(
    pd.DataFrame(
        {new: {old: scores[new].corrwith(scores[old], axis=1).mean() for old in SIGNALS[:5]} for new in CASH}
    ).style.format("{:+.2f}")
)

,cash_flow_yield,accruals
2010-12-31 00:00:00,15%,17%
2011-12-30 00:00:00,45%,51%
2012-12-31 00:00:00,76%,86%
2013-12-31 00:00:00,83%,94%
2014-12-31 00:00:00,83%,93%
2015-12-31 00:00:00,83%,93%
2016-12-30 00:00:00,84%,94%
2017-12-29 00:00:00,83%,95%
2018-12-31 00:00:00,83%,95%
2019-12-31 00:00:00,84%,96%


,"bottom tenth, a year","top tenth, a year",top tenth t
cash_flow_yield,-1.7%,+0.1%,0.0
accruals,+0.3%,+1.7%,0.6


How alike they are to the old scores:


,cash_flow_yield,accruals
value,+0.33,-0.22
quality,+0.29,-0.18
momentum,+0.02,-0.00
low_vol,+0.03,-0.19
issuance,+0.21,-0.02


In [6]:
cash_configs = {
    "+ cash-flow yield": replace(second, factors=(*FIVE, ("cash_flow_yield", 1.0))),
    "+ low accruals": replace(second, factors=(*FIVE, ("accruals", 1.0))),
}
cash_runs = {name: run_strategy(cache, cfg, engine=engine) for name, cfg in cash_configs.items()}
both = {"Second version (notebook 3.2)": runs["Second version (notebook 3.2)"], **cash_runs}
display(pd.DataFrame({name: r.metrics for name, r in both.items()}).T[list(ACTIVE)].style.format(ACTIVE))
print(
    "Registered as research runs",
    [r.run_id for r in cash_runs.values()],
    "- trials of this family so far:",
    max(r.metrics["trials"] for r in cash_runs.values()),
)

,cagr,sharpe,max_drawdown,beta,active_return,tracking_error,information_ratio,deflated_active_sharpe
Second version (notebook 3.2),13.6%,0.87,-34%,0.89,-0.3%,3.7%,-0.14,1%
+ cash-flow yield,13.4%,0.86,-34%,0.89,-0.5%,3.6%,-0.20,1%
+ low accruals,13.9%,0.88,-34%,0.89,-0.1%,3.3%,-0.07,3%


Registered as research runs [49, 50] - trials of this family so far: 12


## Findings

* **Earnings surprise adds nothing here.** Alone, its top decile did no better than the universe average, and in the
  portfolio it left the active return essentially unchanged. Post-earnings-announcement drift is strongest in small
  caps and in the weeks right after the report, and it has weakened in the decades since publication (consistent
  with arbitrage after discovery, McLean and Pontiff 2016); this implementation uses the latest quarter regardless of
  how long ago it was reported.
* **Residual momentum did better in the decile sort than in the portfolio.** Alone, its top decile had the largest
  excess return of all seven scores, equally weighted. In the cap-weighted portfolio it did *worse* than conventional
  momentum, trailing SPTM by about 1 % a year (and still trailing from 2006). The equal-weighted effect did not carry
  over to the large caps that dominate a cap-weighted portfolio - a reminder that many anomalies are concentrated in
  smaller stocks.
* **Every version's worst year is 2023**, 15-18 points behind SPTM: the year a handful of mega-caps drove the market,
  while scores based on cheapness, low volatility and stable profitability held few of them.
* **The cash-flow signals add nothing either.** Alone, free-cash-flow yield's top decile barely beat the universe
  average and low accruals' was within sampling error; in the portfolio, free-cash-flow yield cost a little and low
  accruals changed nothing. Both overlap with what value and quality already measure (correlations of about 0.2-0.3),
  and they cover fewer stocks: banks report no capital expenditure, and before the SEC's filings cover four quarters
  (2010-2011) there is no cash flow at all.

None of the four signals earns a place: the second version of notebook 3.2 remains the best portfolio, and it still adds
no return over SPTM. Every run here is registered, so the next hypothesis is judged against the full count of trials
above.

## Glossary

* **Earnings surprise (SUE)** - standardised unexpected earnings: the latest quarterly earnings relative to what the
  company's own history predicts (the same quarter last year plus its average growth), scaled by the variability of
  that growth.
* **Residual** - the part of a stock's return not explained by its exposure to the market, from a time-series
  regression on the market return.
* **Beta** - the regression slope of a stock's return on the market's: its expected move when the market moves 1 %.
* **Decile** - one of ten equal-sized groups after sorting the cross-section by a score.
* **Active return**, **information ratio**, **deflated Sharpe ratio** - see notebook 3.2.